# 📄 Single-Click Resume Generator

<div style="background-color: #f0f7fb; padding: 20px; border-radius: 10px; margin-bottom: 20px; border-left: 5px solid #3498db;">
    <h2 style="margin-top: 0; color: #3498db;">Simple Resume Generator</h2>
    <p>Just run this cell to automatically:</p>
    <ol>
        <li>Process keywords from your data</li>
        <li>Apply your custom resume structure</li>
        <li>Update your resume with optimized keywords</li>
        <li>Generate a professional PDF</li>
    </ol>
    <p><strong>One click, and you're done!</strong></p>
</div>

In [ ]:
import os
import sys
import time
import subprocess
import datetime
from IPython.display import HTML, display

# Setup paths
base_dir = os.path.abspath('')
scripts_dir = os.path.join(base_dir, 'scripts')
data_dir = os.path.join(base_dir, 'data')
templates_dir = os.path.join(base_dir, 'templates')

# Ensure scripts directory is in path
sys.path.append(base_dir)
sys.path.append(scripts_dir)

# Configure file paths
input_keywords_file = os.path.join(data_dir, "input/lead_gen.csv")
processed_keywords_file = os.path.join(data_dir, "output/processed_keywords.csv")
resume_template = os.path.join(templates_dir, "resume.md")
output_resume_file = os.path.join(data_dir, "output/resume.md")
output_pdf_file = os.path.join(data_dir, "output/resume.pdf")
config_file = os.path.join(templates_dir, "resume_config.yaml")
overrides_file = os.path.join(templates_dir, "overrides.md")

# Styled logging function
def log(message, level="INFO", display_only=False):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if level == "INFO":
        color = "#3498db"  # Blue
        icon = "ℹ️"
    elif level == "SUCCESS":
        color = "#2ecc71"  # Green
        icon = "✅"
    elif level == "WARNING":
        color = "#f39c12"  # Orange
        icon = "⚠️"
    elif level == "ERROR":
        color = "#e74c3c"  # Red
        icon = "❌"
    else:
        color = "#7f8c8d"  # Gray
        icon = "🔹"
    
    html = f"<div style='margin-bottom: 8px;'><span style='color: {color}; font-weight: bold;'>{icon} {level}</span> <span style='color: #7f8c8d; font-size: 0.9em;'>[{timestamp}]</span> {message}</div>"
    display(HTML(html))
    
    if not display_only:
        print(f"[{timestamp}] {level}: {message}")

# Show a section header
def show_section(title):
    display(HTML(f"""
    <div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; margin: 15px 0 5px 0; border-left: 5px solid #6c757d;">
        <h3 style="margin: 0; color: #343a40;">{title}</h3>
    </div>
    """))

# Show a file link with nice formatting
def show_file_link(file_path, description):
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path) / 1024  # KB
        size_str = f"{file_size:.1f} KB" if file_size < 1024 else f"{file_size/1024:.1f} MB"
        
        display(HTML(f"""
        <div style="margin: 10px 0; padding: 10px; border-radius: 5px; background-color: #f8f9fa; display: flex; align-items: center;">
            <div style="margin-right: 15px; font-size: 24px;">📄</div>
            <div>
                <div style="font-weight: bold;">{file_name}</div>
                <div style="color: #6c757d; font-size: 0.9em;">{description} • {size_str}</div>
            </div>
            <div style="margin-left: auto;">
                <a href="{file_path}" target="_blank" style="text-decoration: none; background-color: #007bff; color: white; padding: 5px 10px; border-radius: 3px;">View</a>
            </div>
        </div>
        """))
    else:
        log(f"File not found: {file_path}", "WARNING")

# Run a Python script and capture output
def run_script(script_name, args=None):
    cmd = [sys.executable, os.path.join(scripts_dir, script_name)]
    if args:
        cmd.extend(args)
    
    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = process.communicate()
        
        if process.returncode != 0:
            log(f"Error running {script_name}: {stderr}", "ERROR")
            return False
        
        for line in stdout.split('\n'):
            if line.strip():
                log(line.strip())
        
        return True
    except Exception as e:
        log(f"Exception running {script_name}: {str(e)}", "ERROR")
        return False

# Main execution function
def run_resume_generation():
    # Show welcome message
    display(HTML("""<h2 style="color: #3498db;">🚀 Resume Generation Process</h2>"""))
    log("Starting resume generation process...", "INFO")
    
    try:
        # Check overrides file exists
        if not os.path.exists(overrides_file):
            log(f"Overrides file not found: {overrides_file}", "WARNING")
            log("Creating overrides file with default settings...", "INFO")
            with open(overrides_file, 'w') as f:
                f.write("""# Resume Format Overrides

## Job Titles
JOB_TITLES = [
    "Contract, Remote",
    "6mo. Contract, Remote",
    "12mo. Contract, Remote", 
    "Full-Time, Sandpoint, ID",
    "Jan. 2023",
    "Jan. 2020",
    "Dec. 2019",
    "Aug. 2024 - Current",
    "Jan. 2024 - Jul. 2024",
    "Jan. 2023 - Jan. 2024",
    "Jan. 2019 - Jan. 2023",
    "Feb. 2022"
]

## Capitalizations
CAPITALIZATIONS = [
    "AWS Cloud DevOps",
    "AWS Cloud",
    "Google Cloud Platform",
    "Cloud Services",
    "Cloud and DevOps",
    "Data Engineer",
    "Data Scientist",
    "Data Analyst",
    "Senior Data Engineer",
    "Lead Data Engineer",
    "Senior Data Analyst",
    "Data Engineering Tools",
    "Data Visualization",
    "Big Data"
]""")
            log("Overrides file created successfully", "SUCCESS")
                    
        # Step 1: Process Keywords
        show_section("Step 1: Process Keywords")
        log("Processing keywords from data file...")
        
        if run_script("keywords_processor.py"):
            log("Keywords processed successfully", "SUCCESS")
            show_file_link(processed_keywords_file, "Processed keywords data")
        else:
            log("Failed to process keywords", "ERROR")
            return
        
        # Step 2: Generate Resume Structure
        show_section("Step 2: Apply Resume Structure")
        log("Applying your custom resume structure...")
        
        if run_script("resume_generator.py", ["--config", config_file, "--output", output_resume_file]):
            log("Resume structure applied successfully", "SUCCESS")
            show_file_link(output_resume_file, "Structured resume")
        else:
            log("Failed to apply resume structure", "ERROR")
            return
        
        # Step 3: Update Resume with Keywords
        show_section("Step 3: Update Resume with Keywords")
        log("Enhancing resume with keywords...")
        
        if run_script("update_resume.py", ["--resume", output_resume_file, "--keywords", processed_keywords_file, "--output", output_resume_file]):
            log("Resume updated with keywords successfully", "SUCCESS")
            show_file_link(output_resume_file, "Updated resume with keywords")
        else:
            log("Failed to update resume with keywords", "ERROR")
            return
        
        # Step 4: Generate PDF
        show_section("Step 4: Generate PDF")
        log("Creating professional PDF resume...")
        
        pdf_dir = os.path.dirname(output_pdf_file)
        if run_script("update_resume.py", ["--resume", output_resume_file, "--pdf", "--pdf-dir", pdf_dir]):
            log("PDF generated successfully", "SUCCESS")
            show_file_link(output_pdf_file, "Generated resume PDF")
        else:
            log("Failed to generate PDF", "ERROR")
            return
        
        # Final Summary
        display(HTML("""
        <div style="margin-top: 20px; padding: 15px; border-radius: 5px; background-color: #d4edda; border-left: 5px solid #28a745;">
            <h3 style="margin-top: 0; color: #155724;">✨ Resume Generation Complete</h3>
            <p>Your resume has been successfully generated with optimized keywords and formatting.</p>
        </div>
        """))
        
    except Exception as e:
        log(f"Unexpected error: {str(e)}", "ERROR")

# Run the resume generation process
run_resume_generation()

## 📋 About

<div style="background-color: #f0f0f0; padding: 15px; border-radius: 5px;">
    <h3 style="margin-top: 0;">Simple Resume Generator</h3>
    <p>This tool helps you maintain a keyword-optimized resume that highlights your skills most relevant to job descriptions.</p>
    
    <h4>Key Features</h4>
    <ul>
        <li><strong>Simplified Interface:</strong> Just one button to generate your entire resume</li>
        <li><strong>Smart Keyword Processing:</strong> Analyzes your data to find the most important keywords</li>
        <li><strong>PDF Generation:</strong> Creates a professional PDF ready to send to employers</li>
        <li><strong>Custom Structure:</strong> Uses your resume configuration for personalized section ordering</li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center; color: #777; font-size: 0.9em;">
    <p>Created by Malachi Dunn • © 2024 • <a href="https://github.com/malachi-mindwyre/resume">GitHub Repository</a></p>
</div>